### Description:

Calculate the clustering result of each predicted samples based on the assignment probabilities obtained from retrieve inference for each sample on test set. Then, we could get the populational samples within the same cluster of each test result.  

#### 1. Part1：get assigment of each predited results

In [3]:
import os
import numpy as np
import pandas as pd

In [4]:
def process_test_assigment(test_assignment_path):
   """
   Process assignment probability files and generate classification results.
   
   Args:
       test_assignment_path (str): Path containing 'assign' folder with .npz files.
                                 Each .npz has 'id' and 'z_1' fields for patient ID
                                 and class probabilities.
   
   Returns:
       tuple: (assigment_result, assignment_prob)
           - assigment_result (pd.DataFrame): Top 1-5 predicted classes and probabilities
           - assignment_prob (pd.DataFrame): Raw probabilities for all classes
   """
    
   # Read all npz files
   assign_dir = os.path.join(test_assignment_path, "assign")
   npz_files = [f for f in os.listdir(assign_dir) if f.endswith('.npz')]
   
   # Get first file to determine number of classes
   first_data = np.load(os.path.join(assign_dir, npz_files[0]))
   n_classes = len(first_data['z_1'])
   
   # Initialize dataframes
   prob_cols = [f'cls_{i}_prob' for i in range(n_classes)]
   assignment_prob = pd.DataFrame(columns=['case_name'] + prob_cols)
   assigment_result = pd.DataFrame(columns=['case_name'] + [f'Top{i}-cls' for i in range(1,6)] + [f'Top{i}-prob' for i in range(1,6)])

   # Process each file
   for npz_file in npz_files:
       data = np.load(os.path.join(assign_dir, npz_file))
       case_name = str(data['id'])
       probs = data['z_1']
       
       # Add to assignment_prob
       row_data = {'case_name': case_name}
       row_data.update({f'cls_{i}_prob': probs[i] for i in range(n_classes)})
       assignment_prob = pd.concat([assignment_prob, pd.DataFrame([row_data])], ignore_index=True)
       
       # Get top 5 classes and probabilities
       top5_idx = np.argsort(probs)[-5:][::-1]
       row_data = {
           'case_name': case_name,
           **{f'Top{i+1}-cls': idx for i, idx in enumerate(top5_idx)},
           **{f'Top{i+1}-prob': probs[idx] for i, idx in enumerate(top5_idx)}
       }
       assigment_result = pd.concat([assigment_result, pd.DataFrame([row_data])], ignore_index=True)

   return assigment_result, assignment_prob


In [ ]:

# get assigment for prototype
test_assignment_path = "result_retrieve/PMQM-retrieve-test"
out_test_assigment_path = "retrieve/test_assignment"
os.makedirs(out_test_assigment_path,exist_ok=True)
result_df, prob_df = process_test_assigment(test_assignment_path)

# Save to Excel
result_df.to_excel(os.path.join(out_test_assigment_path, "test_assignment_result.xlsx"), index=False)
prob_df.to_excel(os.path.join(out_test_assigment_path, "test_assignment_prob.xlsx"), index=False)

/tmp/ipykernel_625076/599850518.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  assignment_prob = pd.concat([assignment_prob, pd.DataFrame([row_data])], ignore_index=True)
/tmp/ipykernel_625076/599850518.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  assigment_result = pd.concat([assigment_result, pd.DataFrame([row_data])], ignore_index=True)


#### 2. Part2：get  assigment of each predited results

In [6]:
def get_assign_cluster(result_df, cluster_excel_path, save_path):
    """
    Map each case's Top1-5 classes to their corresponding cluster cases and save results.
    
    Parameters:
    -----------
    result_df : pandas.DataFrame
        DataFrame containing case_name and Top1-5 classification results
    cluster_excel_path : str
        Path to the Excel file containing cluster information for each class
    save_path : str
        Path to save the output Excel file
    """
    
    # Read cluster information from Excel
    cluster_info = {}
    excel = pd.ExcelFile(cluster_excel_path)
    
    # Read each sheet (each class) from the cluster Excel
    for sheet_name in excel.sheet_names:
        if sheet_name.startswith('Class_'):
            class_id = int(sheet_name.split('_')[1])
            df = pd.read_excel(excel, sheet_name)
            cluster_info[class_id] = df['case_name'].tolist()
    
    # Initialize output data structure
    output_data = {}
    
    # Process each case in result_df
    for _, row in result_df.iterrows():
        case_name = row['case_name']
        output_data[case_name] = []
        
        # Process Top1-5 classes
        for i in range(1, 6):
            class_id = int(row[f'Top{i}-cls'])
            if class_id in cluster_info:
                cluster_cases = cluster_info[class_id]
                # Add each case's information
                for cluster_case in cluster_cases:
                    output_data[case_name].append({
                        'Class': class_id,
                        'Top_N': i,
                        'Cluster_Case': cluster_case
                    })
    
    # Create Excel writer
    with pd.ExcelWriter(save_path) as writer:
        # Create a sheet for each case
        for case_name, data in output_data.items():
            # Convert case data to DataFrame
            df = pd.DataFrame(data)
            if not df.empty:
                # Sort by Top_N and Class
                df = df.sort_values(['Top_N', 'Class'])
                # Save to sheet
                df.to_excel(writer, sheet_name=case_name, index=False)
                
                # Adjust column widths
                worksheet = writer.sheets[case_name]
                for idx, col in enumerate(df.columns):
                    max_length = max(
                        df[col].astype(str).apply(len).max(),
                        len(col)
                    ) + 2
                    worksheet.column_dimensions[chr(65 + idx)].width = max_length

In [ ]:
 # by runing this cell, we will have the table of each test sample that contains top1-5 cases name
cluster_excel_path = "retrieve/offlinebank_assigment/cases_of_assignment.xlsx"
save_path = "cluster_files_of_test_sample.xlsx"
get_assign_cluster(result_df, cluster_excel_path, save_path)